In [60]:
import  torch
import torch.nn as nn
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

In [35]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')

In [36]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [37]:
df.drop(columns=['Unnamed: 32', 'id'], inplace=True)

In [38]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [39]:
df.shape

(569, 31)

In [50]:
X_train , X_test, y_train , y_test = train_test_split(df.iloc[:, 1:], df.iloc[: ,0], test_size = 0.2)

In [51]:
scaler = StandardScaler()

In [52]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [53]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

## Conversion

In [65]:
X_train_tensor = torch.from_numpy(X_train.astype('float32'))
X_test_tensor = torch.from_numpy(X_test.astype('float32'))
y_train_tensor = torch.from_numpy(y_train.astype('float32'))
y_test_tensor = torch.from_numpy(y_test.astype('float32'))

## Model Init

In [70]:
class MySimpleNN():

  def __init__(self, X):

    self.weights = torch.rand(X.shape[1], 1, dtype=torch.float32, requires_grad=True)
    self.bias = torch.zeros(1, dtype=torch.float32, requires_grad=True)

  def forward(self, X):
    z = torch.matmul(X, self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_function(self, y_pred, y):
    # Clamp predictions to avoid log(0)
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Calculate loss
    loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
    return loss

In [71]:
learning_rate = 0.1
epochs = 25

 ## Training Pipeline

In [72]:
# create model
model = MySimpleNN(X_train_tensor)

# define loop
for epoch in range(epochs):

  # forward pass
  y_pred = model.forward(X_train_tensor)

  # loss calculate
  loss = model.loss_function(y_pred, y_train_tensor)

  # backward pass
  loss.backward()

  # parameters update
  with torch.no_grad():
    model.weights -= learning_rate * model.weights.grad
    model.bias -= learning_rate * model.bias.grad

  # zero gradients
  model.weights.grad.zero_()
  model.bias.grad.zero_()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 3.354722499847412
Epoch: 2, Loss: 3.217254877090454
Epoch: 3, Loss: 3.0755085945129395
Epoch: 4, Loss: 2.9359025955200195
Epoch: 5, Loss: 2.7927494049072266
Epoch: 6, Loss: 2.635878801345825
Epoch: 7, Loss: 2.4791135787963867
Epoch: 8, Loss: 2.3240373134613037
Epoch: 9, Loss: 2.167243003845215
Epoch: 10, Loss: 2.0152587890625
Epoch: 11, Loss: 1.869386076927185
Epoch: 12, Loss: 1.7167185544967651
Epoch: 13, Loss: 1.5741066932678223
Epoch: 14, Loss: 1.434692621231079
Epoch: 15, Loss: 1.3083159923553467
Epoch: 16, Loss: 1.1972732543945312
Epoch: 17, Loss: 1.1026372909545898
Epoch: 18, Loss: 1.0247224569320679
Epoch: 19, Loss: 0.9627611637115479
Epoch: 20, Loss: 0.9149201512336731
Epoch: 21, Loss: 0.8786852359771729
Epoch: 22, Loss: 0.8513753414154053
Epoch: 23, Loss: 0.8305715918540955
Epoch: 24, Loss: 0.8143417835235596
Epoch: 25, Loss: 0.8012722730636597


In [73]:
model.bias

tensor([-0.1658], requires_grad=True)

In [74]:
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred = (y_pred > 0.9).float()
    accuracy = (y_pred == y_test_tensor).float().mean()
    print(f'Test Accuracy: {accuracy.item() * 100}%')

Test Accuracy: 58.31024646759033%


In [78]:
class Model(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.linear(x)
        out = self.sigmoid(out)
        return out

    def loss_function(self, y_pred, y):
    # Clamp predictions to avoid log(0)
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Calculate loss
        loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
        return loss

In [79]:
model = Model(X_train_tensor.shape[1])

# define loop
for epoch in range(epochs):

  # forward pass
  y_pred = model(X_train_tensor)

  # loss calculate
  loss = model.loss_function(y_pred, y_train_tensor)

  # backward pass
  loss.backward()

  # parameters update
  with torch.no_grad():
    model.linear.weight-= learning_rate * model.linear.weight.grad
    model.linear.bias -= learning_rate * model.linear.bias.grad

  # zero gradients
  model.linear.weight.grad.zero_()
  model.linear.bias.grad.zero_()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.763710081577301
Epoch: 2, Loss: 0.7407563924789429
Epoch: 3, Loss: 0.7269396781921387
Epoch: 4, Loss: 0.7183370590209961
Epoch: 5, Loss: 0.7125527858734131
Epoch: 6, Loss: 0.7082818150520325
Epoch: 7, Loss: 0.7048527002334595
Epoch: 8, Loss: 0.70192551612854
Epoch: 9, Loss: 0.6993269920349121
Epoch: 10, Loss: 0.6969656348228455
Epoch: 11, Loss: 0.6947899460792542
Epoch: 12, Loss: 0.692768931388855
Epoch: 13, Loss: 0.6908816695213318
Epoch: 14, Loss: 0.6891130805015564
Epoch: 15, Loss: 0.6874518394470215
Epoch: 16, Loss: 0.685888409614563
Epoch: 17, Loss: 0.6844145655632019
Epoch: 18, Loss: 0.6830235719680786
Epoch: 19, Loss: 0.6817094087600708
Epoch: 20, Loss: 0.6804667115211487
Epoch: 21, Loss: 0.6792905926704407
Epoch: 22, Loss: 0.6781768798828125
Epoch: 23, Loss: 0.6771214008331299
Epoch: 24, Loss: 0.6761206984519958
Epoch: 25, Loss: 0.6751714944839478


In [83]:
loss_function = nn.BCELoss()

In [81]:
class Model(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.linear(x)
        out = self.sigmoid(out)
        return out


In [84]:
model = Model(X_train_tensor.shape[1])

# define loop
for epoch in range(epochs):

  # forward pass
  y_pred = model(X_train_tensor)

  # loss calculate
  loss = loss_function(y_pred, y_train_tensor.reshape(y_pred.shape))

  # backward pass
  loss.backward()

  # parameters update
  with torch.no_grad():
    model.linear.weight-= learning_rate * model.linear.weight.grad
    model.linear.bias -= learning_rate * model.linear.bias.grad

  # zero gradients
  model.linear.weight.grad.zero_()
  model.linear.bias.grad.zero_()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.8071879148483276
Epoch: 2, Loss: 0.5757684111595154
Epoch: 3, Loss: 0.4626595675945282
Epoch: 4, Loss: 0.39775094389915466
Epoch: 5, Loss: 0.35491251945495605
Epoch: 6, Loss: 0.3240586221218109
Epoch: 7, Loss: 0.3005126714706421
Epoch: 8, Loss: 0.28179243206977844
Epoch: 9, Loss: 0.26644861698150635
Epoch: 10, Loss: 0.253573477268219
Epoch: 11, Loss: 0.24256688356399536
Epoch: 12, Loss: 0.23301470279693604
Epoch: 13, Loss: 0.22462086379528046
Epoch: 14, Loss: 0.21716760098934174
Epoch: 15, Loss: 0.21049058437347412
Epoch: 16, Loss: 0.20446327328681946
Epoch: 17, Loss: 0.19898644089698792
Epoch: 18, Loss: 0.19398097693920135
Epoch: 19, Loss: 0.1893829107284546
Epoch: 20, Loss: 0.18513992428779602
Epoch: 21, Loss: 0.18120868504047394
Epoch: 22, Loss: 0.1775529831647873
Epoch: 23, Loss: 0.17414221167564392
Epoch: 24, Loss: 0.1709505021572113
Epoch: 25, Loss: 0.16795548796653748


In [85]:
y_pred

tensor([[0.0277],
        [0.0095],
        [0.1237],
        [0.0200],
        [0.4060],
        [0.2964],
        [0.1101],
        [0.0745],
        [0.8939],
        [0.4323],
        [0.1078],
        [0.0209],
        [0.1102],
        [0.8541],
        [0.0782],
        [0.1763],
        [0.5542],
        [0.2501],
        [0.8214],
        [0.2831],
        [0.0960],
        [0.1874],
        [0.8146],
        [0.3933],
        [0.9995],
        [0.1326],
        [0.9682],
        [0.7896],
        [0.1873],
        [0.0285],
        [0.2351],
        [0.0051],
        [0.0994],
        [0.0221],
        [0.1948],
        [0.0486],
        [0.1203],
        [0.7325],
        [0.0533],
        [0.1346],
        [0.0316],
        [0.0229],
        [0.1148],
        [0.0318],
        [0.2041],
        [0.1467],
        [0.7901],
        [0.9568],
        [0.2228],
        [0.6813],
        [0.0742],
        [0.9553],
        [0.1206],
        [0.0207],
        [0.3119],
        [0

In [86]:
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred = (y_pred > 0.9).float()
    accuracy = (y_pred == y_test_tensor).float().mean()
    print(f'Test Accuracy: {accuracy.item() * 100}%')

Test Accuracy: 54.77069616317749%


In [87]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [90]:
model = Model(X_train_tensor.shape[1])

# define loop
for epoch in range(epochs):

  # forward pass
  y_pred = model(X_train_tensor)

  # loss calculate
  loss = loss_function(y_pred, y_train_tensor.reshape(y_pred.shape))

  optimizer.zero_grad()

  # backward pass
  loss.backward()

  # parameters update
  optimizer.step()
  # zero gradients


  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.722209632396698
Epoch: 2, Loss: 0.722209632396698
Epoch: 3, Loss: 0.722209632396698
Epoch: 4, Loss: 0.722209632396698
Epoch: 5, Loss: 0.722209632396698
Epoch: 6, Loss: 0.722209632396698
Epoch: 7, Loss: 0.722209632396698
Epoch: 8, Loss: 0.722209632396698
Epoch: 9, Loss: 0.722209632396698
Epoch: 10, Loss: 0.722209632396698
Epoch: 11, Loss: 0.722209632396698
Epoch: 12, Loss: 0.722209632396698
Epoch: 13, Loss: 0.722209632396698
Epoch: 14, Loss: 0.722209632396698
Epoch: 15, Loss: 0.722209632396698
Epoch: 16, Loss: 0.722209632396698
Epoch: 17, Loss: 0.722209632396698
Epoch: 18, Loss: 0.722209632396698
Epoch: 19, Loss: 0.722209632396698
Epoch: 20, Loss: 0.722209632396698
Epoch: 21, Loss: 0.722209632396698
Epoch: 22, Loss: 0.722209632396698
Epoch: 23, Loss: 0.722209632396698
Epoch: 24, Loss: 0.722209632396698
Epoch: 25, Loss: 0.722209632396698


In [91]:
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred = (y_pred > 0.9).float()
    accuracy = (y_pred == y_test_tensor).float().mean()
    print(f'Test Accuracy: {accuracy.item() * 100}%')

Test Accuracy: 58.77193212509155%
